In [3]:
import numpy as np

class SimpleGridworld:
    def __init__(self):
        self.num_states = 5
        self.state = 0

    def reset(self):
        self.state = 0
        return self.state

    def step(self, action):

        if action == 1 and self.state < self.num_states - 1:
            self.state += 1

        if self.state == 4:
            return self.state, 10.0, True
        else:
            return self.state, -1.0, False


In [4]:
def generate_episode(env, policy):
    episode = []
    state = env.reset()
    done = False

    while not done:
        action = np.random.choice(len(policy[state]), p=policy[state])
        next_state, reward, done = env.step(action)
        episode.append((state, action, reward))
        state = next_state

    return episode

def first_visit_mc_prediction(env, policy, num_episodes, gamma=1.0):
    V = np.zeros(env.num_states)
    returns = {s: [] for s in range(env.num_states)}

    for _ in range(num_episodes):
        episode = generate_episode(env, policy)
        G = 0
        visited_states = set()

        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = gamma * G + reward

            state_sequence = [step[0] for step in episode[:t]]
            if state not in state_sequence:
                returns[state].append(G)
                V[state] = np.mean(returns[state])

    return V


In [5]:
if __name__ == "__main__":
    env = SimpleGridworld()

    random_policy = {
        0: [0.2, 0.8],
        1: [0.2, 0.8],
        2: [0.2, 0.8],
        3: [0.2, 0.8],
        4: [0.5, 0.5]
    }

    estimated_values = first_visit_mc_prediction(env, random_policy, num_episodes=5000, gamma=0.9)

    print("Estimated State Values V(s):")
    for state, val in enumerate(estimated_values):
        print(f"State {state}: {val:.4f}")

Estimated State Values V(s):
State 0: 3.2036
State 1: 5.0427
State 2: 7.1304
State 3: 9.5081
State 4: 0.0000
